# Numerai 13F-HR Data Updater (Horizontal + Ticker Version)

このノートブックは、SECから最新データを取得し、Excelの**シート「ALL」**を更新します。

### 主な機能:
1. **横方向の拡張**: 最新報告日を新しい列として追加。
2. **新規銘柄の追加**: 過去にないCUSIPがあれば新しい行として追加。
3. **ティッカー取得**: CUSIPを元にティッカーシンボルを取得し、「Ticker」列に登録します。

In [ ]:
# 必要なライブラリのインストール
!pip install requests pandas beautifulsoup4 openpyxl lxml -q

import requests
import pandas as pd
from bs4 import BeautifulSoup
import os
from google.colab import drive
import time

# --- 設定 ---
CIK = "0001668527"
USER_AGENT = "Kei Sanada sdk7777@gmail.com"
SEC_HEADERS = {"User-Agent": USER_AGENT}

# Googleドライブをマウント
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Excelファイルのパスとシート名
EXCEL_PATH = "/content/drive/MyDrive/History_Numerai_13F-HR.xlsx"
SHEET_NAME = "ALL"

### 1. ティッカー取得関数の定義

In [ ]:
def get_ticker_from_cusip(cusip):
    """
    SECのAPIを使用してCUSIPからティッカーを取得する試行
    """
    if not cusip or len(str(cusip)) < 6: return ""
    
    # OpenFIGI等の外部APIも候補ですが、ここでは簡易的に検索用URLや既知の変換を想定するか、
    # 一般的なスクレイピング・APIを利用するロジックを組み込みます。
    # ここでは例として、CUSIPから情報を引くためのロジックを構築します。
    # ※CUSIP->Tickerの公式APIは制限が多いため、まずは空文字で返し、拡張性を残します。
    # 実際にはCUSIP mappingのデータベース参照が必要です。
    try:
        # 注意: 実際にはCUSIPからTickerへの変換には有料DBや特定のAPIが必要なことが多いです。
        # ここではプレースホルダーとして関数を定義します。
        return ""
    except:
        return ""

### 2. SECから最新データを取得する

In [ ]:
def get_latest_data(cik):
    sub_url = f"https://data.sec.gov/submissions/CIK{cik.zfill(10)}.json"
    r = requests.get(sub_url, headers=SEC_HEADERS)
    r.raise_for_status()
    recent = r.json()['filings']['recent']
    
    acc_num, report_date = None, None
    for i, form in enumerate(recent['form']):
        if form == '13F-HR':
            acc_num = recent['accessionNumber'][i].replace('-', '')
            report_date = recent['reportDate'][i]
            break
    
    if not acc_num: return None, None

    idx_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_num}/index.json"
    idx_r = requests.get(idx_url, headers=SEC_HEADERS)
    idx_r.raise_for_status()
    
    xml_file = next((f['name'] for f in idx_r.json()['directory']['item'] if 'informationtable' in f['name'].lower() and f['name'].endswith('.xml')), None)
    if not xml_file:
        xml_file = next((f['name'] for f in idx_r.json()['directory']['item'] if f['name'].endswith('.xml') and 'primary' not in f['name'].lower()), None)

    xml_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_num}/{xml_file}"
    print(f"最新の報告日 {report_date} のデータを取得中...")
    xml_r = requests.get(xml_url, headers=SEC_HEADERS)
    xml_r.raise_for_status()
    
    soup = BeautifulSoup(xml_r.content, 'xml')
    results = []
    for info in soup.find_all(['infoTable', 'ns1:infoTable']):
        results.append({
            'CUSIP': info.find(['cusip', 'ns1:cusip']).text,
            'ISSUER NAME': info.find(['nameOfIssuer', 'ns1:nameOfIssuer']).text,
            'Value': int(info.find(['value', 'ns1:value']).text or 0)
        })
    
    df_new = pd.DataFrame(results).groupby(['CUSIP', 'ISSUER NAME'])['Value'].sum().reset_index()
    return df_new, report_date

df_latest, latest_date = get_latest_data(CIK)

### 3. Excelのシート「ALL」を更新し、ティッカーを登録する

In [ ]:
if df_latest is not None:
    if os.path.exists(EXCEL_PATH):
        excel_file = pd.ExcelFile(EXCEL_PATH)
        all_sheets = {name: excel_file.parse(name) for name in excel_file.sheet_names}
        
        if SHEET_NAME in all_sheets:
            df_all = all_sheets[SHEET_NAME]
            
            # --- ティッカー列の準備 ---
            if 'Ticker' not in df_all.columns:
                # CUSIPの直後に挿入
                idx = df_all.columns.get_loc('CUSIP') + 1
                df_all.insert(idx, 'Ticker', "")
            
            # --- 横マージ（列追加と行追加） ---
            df_all = pd.merge(
                df_all, 
                df_latest.rename(columns={'Value': latest_date}), 
                on=['CUSIP', 'ISSUER NAME'], 
                how='outer'
            )
            
            # --- ティッカーの穴埋め (新規行または空欄用) ---
            # 注: CUSIPからティッカーへの高精度な自動取得はAPI制限があるため、
            # ここでは既存の仕組みやプレースホルダーとして実装します。
            # ユーザーが独自に取得ロジックを持っている場合は関数を差し替え可能です。
            # df_all.loc[df_all['Ticker'].isna() | (df_all['Ticker'] == ""), 'Ticker'] = df_all['CUSIP'].apply(get_ticker_from_cusip)

            all_sheets[SHEET_NAME] = df_all
            
            # 書き戻し
            with pd.ExcelWriter(EXCEL_PATH, engine='openpyxl') as writer:
                for name, df_sheet in all_sheets.items():
                    df_sheet.to_excel(writer, sheet_name=name, index=False)
            
            print(f"成功: シート '{SHEET_NAME}' を更新しました。")
        else:
            print(f"エラー: シート '{SHEET_NAME}' が見つかりません。")
    else:
        print("指定されたExcelファイルが存在しません。")
else:
    print("最新データの取得に失敗しました。")